In [21]:
import numpy as np
import pandas as pd

print("--- BẮT ĐẦU CHẠY TOÀN BỘ QUY TRÌNH LÀM SẠCH LAB 3 ---")

# =========================================================================
# 1. VẤN ĐỀ 1: Đọc dữ liệu và định nghĩa tiêu đề cột
# =========================================================================
column_names = [
    "Id",
    "Name",
    "Age",
    "Weight",
    "m0006",
    "m0612",
    "m1218",
    "f0006",
    "f0612",
    "f1218",
]  # [cite: 38]

# Đọc file và bỏ qua các dòng lỗi cấu trúc định dạng dấu phẩy
df = pd.read_csv(
    "patient_heart_rate.csv", names=column_names, header=None, on_bad_lines="skip"
)  # [cite: 40]


# =========================================================================
# 2. VẤN ĐỀ 2: Tách cột Name thành Firstname và Lastname
# =========================================================================
if "Name" in df.columns:  # [cite: 44]
    split_names = df["Name"].astype(str).str.split(n=1, expand=True)  # [cite: 47]
    df["Firstname"] = split_names[0]
    df["Lastname"] = split_names[1].fillna("")
    df = df.drop("Name", axis=1)  # [cite: 48]


# =========================================================================
# 3. VẤN ĐỀ 3: Chuẩn hóa cột Weight về đơn vị kg
# =========================================================================
if "Weight" in df.columns:  # [cite: 50]

    def convert_weight(x):
        x = str(x).strip()
        if "lbs" in x:  # [cite: 57]
            val = float(x.replace("lbs", ""))  # [cite: 58, 61]
            return f"{int(val / 2.2)}kgs"  # [cite: 63, 65]
        return x

    df["Weight"] = df["Weight"].apply(convert_weight)


# =========================================================================
# 4. VẤN ĐỀ 4 & 5: Xóa bỏ dòng trống và dòng trùng lặp
# =========================================================================
df.dropna(how="all", inplace=True)  # [cite: 72]

if {"Firstname", "Lastname", "Age", "Weight"}.issubset(df.columns):
    df.drop_duplicates(
        subset=["Firstname", "Lastname", "Age", "Weight"], inplace=True
    )  # [cite: 76]


# =========================================================================
# 5. VẤN ĐỀ 6: Khử nhiễu ký tự không phải ASCII (Non-ASCII)
# =========================================================================
if "Firstname" in df.columns and "Lastname" in df.columns:  # [cite: 77]
    df["Firstname"] = df["Firstname"].replace(
        {r"[^\x00-\x7F]+": ""}, regex=True
    )  # [cite: 83]
    df["Lastname"] = df["Lastname"].replace(
        {r"[^\x00-\x7F]+": ""}, regex=True
    )  # [cite: 83]


# =========================================================================
# 6. VẤN ĐỀ 7: Thống kê và xử lý thiếu dữ liệu (Missing Values) trên Age & Weight
# =========================================================================
if "Age" in df.columns and "Weight" in df.columns:  # [cite: 84]
    df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
    df["Weight"] = pd.to_numeric(
        df["Weight"].astype(str).str.replace("kgs", "", regex=False),
        errors="coerce",
    )

    print(
        f"Số dòng thiếu Age: {df['Age'].isna().sum()} | Thiếu Weight: {df['Weight'].isna().sum()}"
    )  # [cite: 95]

    # Thiếu cả 2 thông tin Age và Weight thì xóa dòng
    df.dropna(subset=["Age", "Weight"], how="all", inplace=True)  # [cite: 96]

    # Gán giá trị thiếu của Age bằng mean
    mean_age = df["Age"].mean()
    df["Age"] = df["Age"].fillna(mean_age)  # [cite: 96]


# =========================================================================
# 7. VẤN ĐỀ 8, 9, 10: Phân rã cột (Melting & Tách biến Sex, Time)
# =========================================================================
# Đảm bảo cột Id ở dạng số chuẩn trước khi phân rã
if "Id" in df.columns:
    df["Id"] = pd.to_numeric(df["Id"], errors="coerce")
    df.dropna(subset=["Id"], inplace=True)
    df["Id"] = df["Id"].astype(int)

# Tiến hành định hình lại DataFrame từ dạng ngang sang dọc (Melting)
df = pd.melt(
    df,
    id_vars=["Id", "Age", "Weight", "Firstname", "Lastname"],
    value_name="PulseRate",
    var_name="sex_and_time",
)  # [cite: 107]

# Thực hiện trích xuất Regex một cách an toàn nhất dựa trên chỉ mục gốc
tmp_df = df["sex_and_time"].str.extract(r"([mf])(\d{2})(\d{2})", expand=True)  # [cite: 109]
tmp_df.columns = ["Sex", "hours_lower", "hours_upper"]  # [cite: 111]
tmp_df["Time"] = tmp_df["hours_lower"] + "-" + tmp_df["hours_upper"]  # [cite: 113]

# CHÌA KHÓA: Reset index của cả 2 bảng độc lập trước khi concat để triệt tiêu lỗi mất cột
df = df.reset_index(drop=True)
tmp_df = tmp_df.reset_index(drop=True)

# Gộp cột kết quả mới vào bảng chính
df = pd.concat([df, tmp_df], axis=1)  # 

# Xóa các cột trung gian thừa
df = df.drop(["sex_and_time", "hours_lower", "hours_upper"], axis=1)  # [cite: 117]
df["PulseRate"] = pd.to_numeric(df["PulseRate"], errors="coerce")


# =========================================================================
# 8. VẤN ĐỀ 11: Khảo sát và xử lý dữ liệu thiếu trên biến huyết áp (PulseRate)
# =========================================================================
missing_ratio = df["PulseRate"].isna().sum() / len(df)  # [cite: 121]
print(f"Tỉ lệ thiếu dữ liệu trên cột huyết áp ban đầu: {missing_ratio:.2%}")


# Thuật toán điền khuyết huyết áp đa tầng y khoa
def impute_pulse_rate(group):
    pulse = group["PulseRate"].copy().reset_index(drop=True)
    for i in range(len(pulse)):
        if pd.isna(pulse[i]):
            # Tầng 1: liền trước & liền sau
            if (
                i > 0
                and i < len(pulse) - 1
                and pd.notna(pulse[i - 1])
                and pd.notna(pulse[i + 1])
            ):
                pulse[i] = (pulse[i - 1] + pulse[i + 1]) / 2  # [cite: 123]
            # Tầng 2: hai giá trị liền trước
            elif i > 1 and pd.notna(pulse[i - 1]) and pd.notna(pulse[i - 2]):
                pulse[i] = (pulse[i - 1] + pulse[i - 2]) / 2  # [cite: 125]
            # Tầng 3: hai giá trị liền sau
            elif (
                i < len(pulse) - 2
                and pd.notna(pulse[i + 1])
                and pd.notna(pulse[i + 2])
            ):
                pulse[i] = (pulse[i + 1] + pulse[i + 2]) / 2  # [cite: 127]
            # Tầng 4: trung bình cá nhân
            elif pulse.notna().any():
                pulse[i] = pulse.mean()  # [cite: 128]
    group["PulseRate"] = pulse.values
    return group


# Thực thi điền khuyết theo nhóm bệnh nhân
df = df.groupby("Id", group_keys=False).apply(impute_pulse_rate)

# Tầng 5: điền theo nhóm giới tính
sex_mean = df.groupby("Sex")["PulseRate"].transform("mean")
df["PulseRate"] = df["PulseRate"].fillna(sex_mean)  # [cite: 129]

# Tầng 6: điền theo trung bình tổng thể hoặc mức ổn định y học (72)
global_mean = df["PulseRate"].mean()
if pd.isna(global_mean):
    global_mean = 72  # [cite: 130]
df["PulseRate"] = df["PulseRate"].fillna(global_mean)  # [cite: 130]


# =========================================================================
# 9. VẤN ĐỀ 12: Rút gọn dữ liệu, Reindex và Lưu trữ dữ liệu sạch
# =========================================================================
# Khôi phục chuỗi hiển thị 'kgs' cho cột cân nặng
df["Weight"] = df["Weight"].apply(
    lambda val: f"{int(val)}kgs" if pd.notna(val) else ""
)

# Sắp xếp cột chuẩn đầu ra (An toàn tuyệt đối bằng việc kiểm tra sự tồn tại của cột)
target_columns = [
    "Id",
    "Age",
    "Weight",
    "Firstname",
    "Lastname",
    "PulseRate",
    "Sex",
    "Time",
]  # [cite: 103, 131]
df = df[[col for col in target_columns if col in df.columns]]

# Reindex lại chỉ số dòng
df.reset_index(drop=True, inplace=True)  # [cite: 131]

# Xuất ra file csv sạch
df.to_csv("patient_heart_rate_clean.csv", index=False)  # [cite: 132]

print("\n--- QUY TRÌNH HOÀN THÀNH, ĐÃ LƯU FILE THÀNH CÔNG! ---")
print(df.head(10))

--- BẮT ĐẦU CHẠY TOÀN BỘ QUY TRÌNH LÀM SẠCH LAB 3 ---
Số dòng thiếu Age: 4 | Thiếu Weight: 4
Tỉ lệ thiếu dữ liệu trên cột huyết áp ban đầu: 51.39%

--- QUY TRÌNH HOÀN THÀNH, ĐÃ LƯU FILE THÀNH CÔNG! ---
    Age Weight Firstname Lastname  PulseRate Sex   Time
0  56.0  70kgs     Micky     Mous  72.000000   m  00-06
1  34.0  70kgs    Donald     Duck  81.666667   m  00-06
2  16.0             Mini    Mouse  68.666667   m  00-06
3  36.1  78kgs   Scrooge   McDuck  78.000000   m  00-06
4  54.0  90kgs      Pink  Panther  72.000000   m  00-06
5  52.0  85kgs      Huey   McDuck  71.666667   m  00-06
6  19.0  56kgs     Dewey   McDuck  74.666667   m  00-06
7  32.0  78kgs      Scpy      Doo  78.000000   m  00-06
8  12.0  45kgs     Louie   McDuck  91.333333   m  00-06
9  36.1  60kgs     Henry      Nam  78.000000   m  00-06
